# Phase 7 — Reporting, Storytelling & Deployment
### Social Services Demand Risk Predictor

---

This is the final phase. A model that cannot be communicated is a model 
that cannot create value. This notebook transforms all prior work into 
outputs that matter to stakeholders, employers, and the public.

**Four deliverables:**

| Deliverable | Tool | Audience |
|---|---|---|
| A | Executive summary notebook | Non-technical policy audience |
| B | Plotly Dash dashboard | Interactive exploration |
| C | Model card | Technical accountability |
| D | README + GitHub publish | Portfolio / hiring managers |

**Inputs:** All outputs from Phases 1–6  
**Outputs:**  
- `reports/executive_summary.html`  
- `app/dashboard.py`  
- `MODEL_CARD.md`  
- `README.md` (final)  
- GitHub repository ready to publish  

**Libraries:** plotly, dash, jinja2, joblib

## 1. Setup — Imports & Paths

In [ ]:
import os
import json
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime
import joblib

# ── Plotly ────────────────────────────────────────────────────────────────
try:
    import plotly.express       as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
    import plotly
    print(f"  plotly  : {plotly.__version__}")
except ImportError:
    PLOTLY_AVAILABLE = False
    print("  plotly not installed — run: pip install plotly")

# ── Dash ──────────────────────────────────────────────────────────────────
try:
    import dash
    from dash import dcc, html, Input, Output, callback
    DASH_AVAILABLE = True
    print(f"  dash    : {dash.__version__}")
except ImportError:
    DASH_AVAILABLE = False
    print("  dash not installed — run: pip install dash")

# ── Plot style ─────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi"       : 120,
    "figure.facecolor" : "white",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
})

print(f"  pandas  : {pd.__version__}")
print(f"  numpy   : {np.__version__}")
print()
print("Setup complete.")

In [ ]:
# Paths

NOTEBOOK_DIR  = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT  = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
MODELS_DIR    = os.path.join(PROJECT_ROOT, "models")
REPORTS_DIR   = os.path.join(PROJECT_ROOT, "reports")
APP_DIR       = os.path.join(PROJECT_ROOT, "app")

os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(APP_DIR,     exist_ok=True)

print("Paths configured:")
print(f"  Project root : {PROJECT_ROOT}")
print(f"  Processed    : {PROCESSED_DIR}")
print(f"  Models       : {MODELS_DIR}")
print(f"  Reports      : {REPORTS_DIR}")
print(f"  App          : {APP_DIR}")

## 2. Load All Phase Outputs

In [ ]:
def load_csv_safe(filename, label):
    path = os.path.join(PROCESSED_DIR, filename)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  [✓] {label:<35} {df.shape[0]:>6,} rows x {df.shape[1]:>3} cols")
        return df
    else:
        print(f"  [✗] {label:<35} not found — {filename}")
        return None

def load_model_safe(filename, label):
    path = os.path.join(MODELS_DIR, filename)
    if os.path.exists(path):
        model = joblib.load(path)
        size  = os.path.getsize(path) / 1024
        print(f"  [✓] {label:<35} {size:>8.1f} KB")
        return model
    else:
        print(f"  [✗] {label:<35} not found — {filename}")
        return None

print("Loading processed datasets...")
print("-" * 55)
df_seifa      = load_csv_safe("seifa_with_risk_tier.csv",   "SEIFA with risk tiers")
df_sentiment  = load_csv_safe("sentiment_results.csv",      "Sentiment results")
df_topics     = load_csv_safe("topic_model_results.csv",    "Topic model results")
df_comparison = load_csv_safe("model_comparison.csv",       "Model comparison")
X_test        = load_csv_safe("X_test.csv",                 "X test")
y_test_df     = load_csv_safe("y_test.csv",                 "y test")

print()
print("Loading trained models...")
print("-" * 55)
pipeline  = load_model_safe("preprocessing_pipeline.joblib", "Preprocessing pipeline")
rf_model  = load_model_safe("random_forest.joblib",          "Random Forest")
xgb_model = load_model_safe("xgboost_model.joblib",         "XGBoost")

print()
print("Load complete.")

## 3. Deliverable A — Executive Summary

The executive summary tells the complete story of the project 
in plain language with supporting visuals. It contains no raw 
code — only findings, charts, and recommendations.

This is the document you would hand to a Deputy Secretary, 
a portfolio review committee, or a hiring manager.

### 3a. Key Findings Chart — Risk Tier Distribution

In [ ]:
if df_seifa is not None and PLOTLY_AVAILABLE:
    risk_counts = df_seifa["risk_tier"].value_counts().reset_index()
    risk_counts.columns = ["Risk Tier", "SA2 Regions"]
    risk_counts["Risk Tier"] = pd.Categorical(
        risk_counts["Risk Tier"],
        categories=["High", "Medium", "Low"], ordered=True
    )
    risk_counts = risk_counts.sort_values("Risk Tier")

    color_map = {"High": "#993C1D", "Medium": "#BA7517", "Low": "#1D9E75"}

    fig = px.bar(
        risk_counts,
        x     = "Risk Tier",
        y     = "SA2 Regions",
        color = "Risk Tier",
        color_discrete_map = color_map,
        text  = "SA2 Regions",
        title = "SA2 Regions by Welfare Demand Risk Tier",
    )
    fig.update_traces(texttemplate="%{text:,}", textposition="outside")
    fig.update_layout(
        showlegend   = False,
        plot_bgcolor = "white",
        paper_bgcolor= "white",
        font         = dict(family="Arial", size=13),
        yaxis_title  = "Number of SA2 Regions",
        xaxis_title  = "Risk Tier",
        title_font_size = 16,
    )
    fig.show()
    fig.write_html(os.path.join(REPORTS_DIR, "34_risk_tier_interactive.html"))
    print("[✓] Saved: reports/34_risk_tier_interactive.html")

elif df_seifa is not None:
    # Matplotlib fallback
    counts = df_seifa["risk_tier"].value_counts()
    colors = {"High": "#993C1D", "Medium": "#BA7517", "Low": "#1D9E75"}
    fig, ax = plt.subplots(figsize=(7, 5))
    for tier in ["High", "Medium", "Low"]:
        if tier in counts:
            ax.bar(tier, counts[tier], color=colors[tier], edgecolor="white")
    ax.set_title("SA2 Regions by Risk Tier", fontweight="bold")
    ax.set_ylabel("Number of SA2 Regions")
    plt.tight_layout()
    fig.savefig(os.path.join(REPORTS_DIR, "34_risk_tier_bar.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print("[✓] Saved: reports/34_risk_tier_bar.png")

### 3b. Model Performance Summary Chart

In [ ]:
if df_comparison is not None and PLOTLY_AVAILABLE:
    fig = go.Figure()

    fig.add_trace(go.Bar(
        name = "F1 Macro",
        x    = df_comparison["Model"],
        y    = df_comparison["F1 Macro"],
        marker_color = "#1D9E75",
        text = df_comparison["F1 Macro"].round(3),
        textposition = "outside"
    ))

    fig.add_trace(go.Bar(
        name = "F1 Weighted",
        x    = df_comparison["Model"],
        y    = df_comparison["F1 Weighted"],
        marker_color = "#534AB7",
        text = df_comparison["F1 Weighted"].round(3),
        textposition = "outside"
    ))

    fig.add_hline(y=0.70, line_dash="dash", line_color="#993C1D",
                  annotation_text="Target F1 ≥ 0.70",
                  annotation_position="top right")

    fig.update_layout(
        title        = "Classification Model Comparison — F1 Scores",
        barmode      = "group",
        plot_bgcolor = "white",
        paper_bgcolor= "white",
        font         = dict(family="Arial", size=13),
        yaxis        = dict(title="F1 Score", range=[0, 1.1]),
        xaxis_title  = "Model",
        legend       = dict(orientation="h", yanchor="bottom",
                            y=1.02, xanchor="right", x=1),
        title_font_size = 16,
    )
    fig.show()
    fig.write_html(os.path.join(REPORTS_DIR, "35_model_comparison_interactive.html"))
    print("[✓] Saved: reports/35_model_comparison_interactive.html")

### 3c. Sentiment Trend Chart (Interactive)

In [ ]:
if df_sentiment is not None and PLOTLY_AVAILABLE:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x    = df_sentiment["year"],
        y    = df_sentiment["compound"],
        mode = "lines+markers+text",
        name = "Compound Sentiment",
        line = dict(color="#534AB7", width=2.5),
        marker = dict(size=8),
        text = df_sentiment["compound"].round(3).astype(str),
        textposition = "top center"
    ))

    fig.add_hrect(y0=0.05,  y1=1.0,   fillcolor="#1D9E75",
                  opacity=0.07, line_width=0)
    fig.add_hrect(y0=-1.0,  y1=-0.05, fillcolor="#993C1D",
                  opacity=0.07, line_width=0)
    fig.add_hline(y=0, line_dash="dot", line_color="gray", line_width=1)

    fig.update_layout(
        title       = "VADER Sentiment Score — DSS Annual Reports 2018–2023<br>"
                      "<sub>Green zone = positive · Red zone = negative</sub>",
        xaxis_title = "Year",
        yaxis_title = "Compound Sentiment Score",
        plot_bgcolor = "white",
        paper_bgcolor= "white",
        font        = dict(family="Arial", size=13),
        title_font_size = 16,
        yaxis       = dict(range=[-0.5, 0.5]),
    )
    fig.show()
    fig.write_html(os.path.join(REPORTS_DIR, "36_sentiment_interactive.html"))
    print("[✓] Saved: reports/36_sentiment_interactive.html")

### 3d. Export Full Executive Summary as HTML

In [ ]:
# Build executive summary HTML report
# This embeds all key findings and charts into a single self-contained file

today     = datetime.today().strftime("%B %Y")
best_model = df_comparison.iloc[0]["Model"] if df_comparison is not None else "Best model"
best_f1    = float(df_comparison.iloc[0]["F1 Macro"]) if df_comparison is not None else 0.0
n_high     = int((df_seifa["risk_tier"] == "High").sum()) if df_seifa is not None else 0
n_regions  = len(df_seifa) if df_seifa is not None else 0

html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Social Services Demand Risk Predictor — Executive Summary</title>
<style>
  body        {{ font-family: Arial, sans-serif; max-width: 900px;
                margin: 0 auto; padding: 2rem; color: #1a1a1a; line-height: 1.7; }}
  h1          {{ color: #1D9E75; border-bottom: 3px solid #1D9E75;
                padding-bottom: 0.5rem; }}
  h2          {{ color: #185FA5; margin-top: 2.5rem; }}
  h3          {{ color: #534AB7; }}
  .badge      {{ display: inline-block; padding: 4px 12px; border-radius: 6px;
                font-weight: bold; font-size: 0.9rem; margin: 2px; }}
  .badge-high {{ background: #FAECE7; color: #993C1D; }}
  .badge-med  {{ background: #FAEEDA; color: #BA7517; }}
  .badge-low  {{ background: #E1F5EE; color: #1D9E75; }}
  .metric-row {{ display: flex; gap: 1rem; margin: 1.5rem 0;
                flex-wrap: wrap; }}
  .metric     {{ background: #F5F5F0; border-radius: 8px; padding: 1rem 1.5rem;
                flex: 1; min-width: 160px; }}
  .metric-val {{ font-size: 2rem; font-weight: bold; color: #1D9E75; }}
  .metric-lbl {{ font-size: 0.85rem; color: #5F5E5A; margin-top: 4px; }}
  table       {{ width: 100%; border-collapse: collapse; margin: 1rem 0; }}
  th          {{ background: #1D9E75; color: white; padding: 10px 14px;
                text-align: left; }}
  td          {{ padding: 8px 14px; border-bottom: 1px solid #E0E0E0; }}
  tr:nth-child(even) {{ background: #F9F9F6; }}
  .finding    {{ border-left: 4px solid #1D9E75; padding: 0.5rem 1rem;
                margin: 0.8rem 0; background: #F9FFF9; border-radius: 0 6px 6px 0; }}
  .warning    {{ border-left: 4px solid #BA7517; padding: 0.5rem 1rem;
                margin: 0.8rem 0; background: #FFFBF0; border-radius: 0 6px 6px 0; }}
  .footer     {{ margin-top: 3rem; padding-top: 1rem;
                border-top: 1px solid #E0E0E0; color: #888; font-size: 0.85rem; }}
  .tag        {{ display: inline-block; background: #E6F1FB; color: #185FA5;
                padding: 2px 8px; border-radius: 4px; font-size: 0.8rem;
                margin: 2px; }}
</style>
</head>
<body>

<h1>Social Services Demand Risk Predictor</h1>
<p><strong>Executive Summary</strong> &nbsp;|&nbsp; Australian Public Sector
&nbsp;|&nbsp; Department of Social Services &nbsp;|&nbsp; {today}</p>
<p>
  <span class="tag">Python</span>
  <span class="tag">Machine Learning</span>
  <span class="tag">Time-Series Forecasting</span>
  <span class="tag">NLP</span>
  <span class="tag">Open Government Data</span>
</p>

<h2>Project Overview</h2>
<p>This project applies end-to-end data science to a genuine Australian
public sector challenge: identifying which communities face the highest
risk of increased demand for social services. Using entirely open,
publicly available data from ABS, DSS, and AIHW, the project delivers
three analytical outputs that directly support evidence-based policy.</p>

<div class="metric-row">
  <div class="metric">
    <div class="metric-val">{n_regions:,}</div>
    <div class="metric-lbl">SA2 regions analysed</div>
  </div>
  <div class="metric">
    <div class="metric-val">{n_high:,}</div>
    <div class="metric-lbl">High-risk regions identified</div>
  </div>
  <div class="metric">
    <div class="metric-val">{best_f1:.2f}</div>
    <div class="metric-lbl">Best model F1 score</div>
  </div>
  <div class="metric">
    <div class="metric-val">6</div>
    <div class="metric-lbl">Policy documents analysed</div>
  </div>
</div>

<h2>Research Questions & Outcomes</h2>
<table>
  <tr><th>Question</th><th>Technique</th><th>Key Result</th></tr>
  <tr>
    <td>Which SA2 regions face highest welfare demand risk?</td>
    <td>Classification (XGBoost)</td>
    <td>F1 macro = {best_f1:.3f} on held-out test set</td>
  </tr>
  <tr>
    <td>How will demand trend over the next 6–12 months?</td>
    <td>Prophet + ARIMA forecasting</td>
    <td>4-quarter forecast with 95% confidence intervals</td>
  </tr>
  <tr>
    <td>What themes dominate recent social policy documents?</td>
    <td>LDA topic modelling + VADER sentiment</td>
    <td>6 key policy topics tracked 2018–2023</td>
  </tr>
</table>

<h2>Key Findings</h2>

<div class="finding">
  <strong>Geographic concentration of disadvantage:</strong> High-risk SA2 regions
  are concentrated in the Northern Territory, remote Western Australia, and outer
  suburban corridors of major cities — consistent with ABS SEIFA patterns.
</div>

<div class="finding">
  <strong>IRSD is the strongest predictor:</strong> SHAP analysis confirmed that
  the IRSD socio-economic disadvantage score is the single most influential feature
  across all classification models, followed by remoteness classification and
  population size.
</div>

<div class="finding">
  <strong>COVID-19 visible in forecasting data:</strong> The 2020 quarterly series
  shows a sharp spike in recipients coinciding with the coronavirus supplement —
  the Prophet model captures this as a structural break in trend.
</div>

<div class="finding">
  <strong>Policy language shifted post-2020:</strong> LDA topic modelling detected
  a surge in economic support and emergency response language in 2020–2021 annual
  reports, with sentiment scores dipping before recovering in 2022–2023.
</div>

<div class="warning">
  <strong>Model limitation:</strong> The classification model is trained on
  cross-sectional data (one point in time). It does not capture dynamic changes
  within regions. Temporal validation with future DSS payment data is recommended
  before operational use.
</div>

<h2>Data Sources</h2>
<table>
  <tr><th>Dataset</th><th>Provider</th><th>Licence</th></tr>
  <tr><td>SEIFA 2021 (SA2 level)</td><td>Australian Bureau of Statistics</td>
      <td>CC BY 4.0</td></tr>
  <tr><td>DSS Payment Demographic Data</td><td>Department of Social Services</td>
      <td>CC BY 4.0</td></tr>
  <tr><td>Regional Population 2022</td><td>Australian Bureau of Statistics</td>
      <td>CC BY 4.0</td></tr>
  <tr><td>DSS Annual Reports 2018–2023</td><td>Department of Social Services</td>
      <td>CC BY 4.0</td></tr>
</table>

<h2>Technical Stack</h2>
<p>
  <span class="tag">Python 3.11</span>
  <span class="tag">pandas</span>
  <span class="tag">scikit-learn</span>
  <span class="tag">XGBoost</span>
  <span class="tag">SHAP</span>
  <span class="tag">Prophet</span>
  <span class="tag">statsmodels</span>
  <span class="tag">spaCy</span>
  <span class="tag">gensim (LDA)</span>
  <span class="tag">VADER</span>
  <span class="tag">Plotly</span>
  <span class="tag">Dash</span>
  <span class="tag">JupyterLab</span>
</p>

<div class="footer">
  <p>Generated: {datetime.today().strftime("%d %B %Y")} &nbsp;|&nbsp;
  Repository: aus-social-services-ds &nbsp;|&nbsp;
  All data sourced from open Australian Government datasets under CC BY 4.0</p>
</div>

</body>
</html>
"""

summary_path = os.path.join(REPORTS_DIR, "executive_summary.html")
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"[✓] Executive summary saved → {summary_path}")
print()
print("To view: open reports/executive_summary.html in your browser")

## 4. Deliverable B — Plotly Dash Dashboard

The dashboard provides an interactive 3-panel exploration tool 
for stakeholders. We write it as a standalone Python script 
(`app/dashboard.py`) that can be run independently of JupyterLab.

The three panels are:
1. **Risk Map** — SA2 regions coloured by risk tier
2. **Forecast Chart** — quarterly demand trend with controls
3. **Topic Trends** — LDA topic proportions over time

In [ ]:
# Write the Dash dashboard as a standalone Python script

dashboard_code = '''import os
import pandas as pd
import numpy as np
import plotly.express       as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dash import Dash, dcc, html, Input, Output
import warnings
warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────
SCRIPT_DIR    = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT  = os.path.abspath(os.path.join(SCRIPT_DIR, ".."))
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

# ── Load data ──────────────────────────────────────────────────────────────
def safe_load(filename):
    path = os.path.join(PROCESSED_DIR, filename)
    return pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()

df_seifa     = safe_load("seifa_with_risk_tier.csv")
df_sentiment = safe_load("sentiment_results.csv")
df_topics    = safe_load("topic_model_results.csv")

COLOR_MAP = {"High": "#993C1D", "Medium": "#BA7517", "Low": "#1D9E75"}

# ── App layout ─────────────────────────────────────────────────────────────
app = Dash(__name__, title="Social Services Risk Dashboard")

app.layout = html.Div([

    html.Div([
        html.H1("Social Services Demand Risk Predictor",
                style={"color": "#1D9E75", "marginBottom": "4px"}),
        html.P("Australian Public Sector — Department of Social Services",
               style={"color": "#5F5E5A", "marginTop": "0"}),
    ], style={"padding": "1.5rem 2rem 0.5rem", "borderBottom": "2px solid #E0E0E0"}),

    html.Div([

        # ── Panel 1: Risk Tier Distribution ────────────────────────────────
        html.Div([
            html.H3("SA2 Risk Tier Distribution",
                    style={"color": "#185FA5", "marginBottom": "8px"}),
            html.P("Regions classified as High / Medium / Low welfare demand risk "
                   "based on IRSD socioeconomic disadvantage score.",
                   style={"color": "#5F5E5A", "fontSize": "13px"}),
            dcc.Graph(id="risk-tier-chart", style={"height": "380px"}),

            html.Div([
                html.Label("Filter by State / Territory:",
                           style={"fontWeight": "bold", "fontSize": "13px"}),
                dcc.Dropdown(
                    id      = "state-filter",
                    options = (
                        [{"label": "All States", "value": "All"}] +
                        [{"label": s, "value": s}
                         for s in sorted(df_seifa.get(
                             next((c for c in df_seifa.columns
                                   if "state" in c.lower()), ""), pd.Series()
                         ).dropna().unique())]
                    ) if len(df_seifa) > 0 else [{"label": "All", "value": "All"}],
                    value      = "All",
                    clearable  = False,
                    style      = {"marginTop": "6px"}
                ),
            ], style={"marginTop": "12px"}),

        ], style={"background": "white", "borderRadius": "10px",
                  "padding": "1.2rem", "border": "1px solid #E0E0E0",
                  "marginBottom": "1rem"}),

        # ── Panel 2: Sentiment Trend ───────────────────────────────────────
        html.Div([
            html.H3("Policy Sentiment Trend 2018–2023",
                    style={"color": "#185FA5", "marginBottom": "8px"}),
            html.P("VADER compound sentiment score from DSS Annual Reports. "
                   "Positive = constructive language. Negative = urgent/crisis language.",
                   style={"color": "#5F5E5A", "fontSize": "13px"}),
            dcc.Graph(id="sentiment-chart", style={"height": "340px"}),
        ], style={"background": "white", "borderRadius": "10px",
                  "padding": "1.2rem", "border": "1px solid #E0E0E0",
                  "marginBottom": "1rem"}),

        # ── Panel 3: Topic Evolution ───────────────────────────────────────
        html.Div([
            html.H3("Policy Topic Evolution Over Time",
                    style={"color": "#185FA5", "marginBottom": "8px"}),
            html.P("LDA topic proportions per year. Each colour represents "
                   "a distinct policy theme discovered from DSS Annual Reports.",
                   style={"color": "#5F5E5A", "fontSize": "13px"}),
            dcc.Graph(id="topic-chart", style={"height": "340px"}),
        ], style={"background": "white", "borderRadius": "10px",
                  "padding": "1.2rem", "border": "1px solid #E0E0E0",
                  "marginBottom": "1rem"}),

    ], style={"padding": "1rem 2rem", "maxWidth": "1000px", "margin": "0 auto"}),

    html.Div([
        html.P("Data sources: ABS SEIFA 2021 · DSS Payment Data · DSS Annual Reports "
               "2018–2023 · All open data under CC BY 4.0",
               style={"color": "#9E9E9E", "fontSize": "12px", "textAlign": "center"})
    ], style={"padding": "1rem", "borderTop": "1px solid #E0E0E0", "marginTop": "1rem"}),

], style={"fontFamily": "Arial, sans-serif", "background": "#F8F8F5",
          "minHeight": "100vh"})


# ── Callbacks ──────────────────────────────────────────────────────────────
@app.callback(
    Output("risk-tier-chart", "figure"),
    Input("state-filter", "value")
)
def update_risk_chart(selected_state):
    if len(df_seifa) == 0:
        return go.Figure().add_annotation(text="No data loaded",
                                          showarrow=False, font_size=16)

    state_col = next((c for c in df_seifa.columns if "state" in c.lower()), None)
    df = df_seifa.copy()

    if selected_state != "All" and state_col:
        df = df[df[state_col] == selected_state]

    counts = df["risk_tier"].value_counts().reindex(
        ["High", "Medium", "Low"], fill_value=0
    ).reset_index()
    counts.columns = ["Risk Tier", "Count"]

    fig = px.bar(
        counts, x="Risk Tier", y="Count",
        color="Risk Tier",
        color_discrete_map=COLOR_MAP,
        text="Count"
    )
    fig.update_traces(texttemplate="%{text:,}", textposition="outside")
    fig.update_layout(
        showlegend=False, plot_bgcolor="white", paper_bgcolor="white",
        margin=dict(t=20, b=20), yaxis_title="SA2 Regions"
    )
    return fig


@app.callback(
    Output("sentiment-chart", "figure"),
    Input("state-filter", "value")   # dummy input to wire the callback
)
def update_sentiment_chart(_):
    if len(df_sentiment) == 0:
        return go.Figure().add_annotation(text="No sentiment data loaded",
                                          showarrow=False, font_size=16)
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_sentiment["year"], y=df_sentiment["compound"],
        mode="lines+markers+text",
        line=dict(color="#534AB7", width=2.5),
        marker=dict(size=8),
        text=df_sentiment["compound"].round(3).astype(str),
        textposition="top center",
        name="Compound score"
    ))
    fig.add_hrect(y0=0.05,  y1=1,   fillcolor="#1D9E75", opacity=0.08, line_width=0)
    fig.add_hrect(y0=-1,    y1=-0.05, fillcolor="#993C1D", opacity=0.08, line_width=0)
    fig.add_hline(y=0, line_dash="dot", line_color="gray", line_width=1)
    fig.update_layout(
        plot_bgcolor="white", paper_bgcolor="white",
        margin=dict(t=20, b=20),
        yaxis=dict(title="Compound score", range=[-0.5, 0.5]),
        xaxis_title="Year"
    )
    return fig


@app.callback(
    Output("topic-chart", "figure"),
    Input("state-filter", "value")   # dummy input
)
def update_topic_chart(_):
    if len(df_topics) == 0:
        return go.Figure().add_annotation(text="No topic data loaded",
                                          showarrow=False, font_size=16)

    topic_cols = [c for c in df_topics.columns if c != "year"]
    year_col   = "year" if "year" in df_topics.columns else df_topics.index.name

    df_t = df_topics.reset_index() if year_col == df_topics.index.name else df_topics
    colors = ["#1D9E75","#534AB7","#BA7517","#993C1D",
              "#185FA5","#3B6D11","#5F5E5A","#D4537E"]

    fig = go.Figure()
    for i, col in enumerate(topic_cols):
        fig.add_trace(go.Bar(
            name=col, x=df_t.get("year", df_t.index),
            y=df_t[col], marker_color=colors[i % len(colors)]
        ))

    fig.update_layout(
        barmode="stack", plot_bgcolor="white", paper_bgcolor="white",
        margin=dict(t=20, b=20),
        yaxis=dict(title="Topic proportion",
                   tickformat=".0%", range=[0, 1]),
        xaxis_title="Year",
        legend=dict(orientation="v", x=1.01, y=1, font_size=9)
    )
    return fig


if __name__ == "__main__":
    print("Starting Social Services Risk Dashboard...")
    print("Open your browser at: http://127.0.0.1:8050")
    app.run(debug=True, port=8050)
'''

dashboard_path = os.path.join(APP_DIR, "dashboard.py")
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(dashboard_code)

print(f"[✓] Dashboard saved → {dashboard_path}")
print()
print("To run the dashboard:")
print(f"  cd {APP_DIR}")
print("  python dashboard.py")
print("  Then open http://127.0.0.1:8050 in your browser")

## 5. Deliverable C — Model Card

A model card documents what the model does, how it performs, 
its limitations, and how it should and should not be used.

This is mandatory in responsible AI practice and increasingly 
required by Australian Government agencies. Having one in your 
portfolio demonstrates maturity as a data scientist.

In [ ]:
today_str  = datetime.today().strftime("%d %B %Y")
best_model = df_comparison.iloc[0]["Model"] if df_comparison is not None else "Best model"
best_f1    = float(df_comparison.iloc[0]["F1 Macro"]) if df_comparison is not None else 0.0

model_card_content = f"""# Model Card — Social Services Demand Risk Predictor

**Model name:** Welfare Demand Risk Classifier  
**Model type:** Multi-class classification  
**Version:** 1.0  
**Date:** {today_str}  
**Author:** Firoz Mahmud 
**Email:** fmahmud.ruet@gmail.com
**Repository:** aus-social-services-ds  

---

## Model Description

This model classifies Australian Statistical Area Level 2 (SA2) regions 
into three welfare demand risk tiers — High, Medium, and Low — based on 
socioeconomic, geographic, and demographic features derived from publicly 
available ABS and DSS data.

It is intended as a **research and portfolio tool** to demonstrate 
end-to-end data science applied to Australian public sector problems. 
It is not intended for operational use in welfare decision-making.

---

## Intended Use

### Intended uses
- Portfolio demonstration of end-to-end DS methodology
- Research into geographic patterns of welfare demand
- Educational example of responsible ML in government contexts
- Starting point for further analysis with richer longitudinal data

### Out-of-scope uses
- Direct allocation or denial of welfare payments to individuals
- Automated eligibility assessments
- Profiling of individual citizens
- Any use that treats model output as ground truth without expert review

---

## Training Data

| Dataset | Source | Version | Licence |
|---|---|---|---|
| ABS SEIFA 2021 (SA2) | Australian Bureau of Statistics | 2021 Census | CC BY 4.0 |
| DSS Payment Demographic Data | Department of Social Services | Jun 2023 | CC BY 4.0 |
| ABS Regional Population | Australian Bureau of Statistics | 2021-22 | CC BY 4.0 |

**No individual-level data was used.** All data is aggregated at SA2 
region level (3,000–25,000 residents per region).

---

## Model Performance

| Metric | Value | Target | Met? |
|---|---|---|---|
| F1 Score (macro) | {best_f1:.3f} | ≥ 0.70 | {'✓' if isinstance(best_f1, float) and best_f1 >= 0.70 else '✗'} |
| F1 Score (weighted) | See report | ≥ 0.70 | — |
| ROC-AUC (macro OVR) | See report | ≥ 0.80 | — |

Evaluation was performed on a held-out test set (15% of data) 
that was not used during training or hyperparameter tuning.

---

## Features Used

| Feature | Description | Source |
|---|---|---|
| irsd_score | Index of Relative Socio-economic Disadvantage | ABS SEIFA |
| irsad_score | Index of Relative Socio-economic Advantage & Disadvantage | ABS SEIFA |
| irsd_decile | IRSD decile (1–10, 1 = most disadvantaged) | ABS SEIFA |
| log_population | Log-transformed usual resident population | ABS SEIFA |
| score_range | Difference between IRSAD and IRSD scores | Engineered |
| disadvantage_flag | Binary: 1 if IRSD decile ≤ 3 | Engineered |
| remoteness_encoded | Urban=0, Regional=1, Remote=2 | Engineered |
| state_encoded | Ordinal encoding of state/territory | ABS SEIFA |

---

## Limitations & Risks

### Known limitations
1. **Cross-sectional only** — trained on a single point in time. 
   Does not capture within-region change dynamics over time.

2. **Geographic aggregation** — SA2 regions mask significant 
   intra-region variation. High-risk individuals may live in 
   Low-risk regions and vice versa.

3. **Feature proxies** — remoteness and state are imperfect proxies 
   for the complex social factors driving welfare demand.

4. **Class imbalance** — the model was trained with SMOTE oversampling 
   and balanced class weights. Performance on the minority (High-risk) 
   class may be lower in deployment on new data.

### Ethical considerations
1. **Stigmatisation risk** — labelling communities as "High risk" 
   could be misused to stigmatise regions rather than to direct support.
   The model output should always be framed as "likely to benefit from 
   additional support", not "likely to be a burden".

2. **Indigenous communities** — a disproportionate share of High-risk 
   regions are in remote areas with significant Indigenous populations. 
   Any use of this model in Indigenous policy contexts must involve 
   consultation with affected communities and Indigenous data governance 
   frameworks (AIATSIS Code of Ethics).

3. **Self-fulfilling prophecy** — if resource allocation follows model 
   predictions, regions predicted as Low-risk may receive less support, 
   which could cause their situation to deteriorate — a feedback loop 
   that the model cannot detect.

---

## Recommendations for Safe Use

- Always combine model outputs with local knowledge and qualitative context
- Treat predictions as one input among many, not as a decision
- Re-train annually as new DSS payment data and ABS updates become available
- Commission an independent bias audit before any operational deployment
- Establish a human review process for any High-risk classification that 
  informs a policy decision

---

## Caveats

This model was developed as a **portfolio project** using open data. 
The author is not responsible for any decisions made using its outputs. 
The model has not undergone external validation or independent audit.

---

*Model card template inspired by Mitchell et al. (2019) and the 
Australian Government AI Ethics Framework (2024).*
"""

model_card_path = os.path.join(PROJECT_ROOT, "MODEL_CARD.md")
with open(model_card_path, "w", encoding="utf-8") as f:
    f.write(model_card_content)

print(f"[✓] Model card saved → {model_card_path}")
print()
# Print a preview
lines = model_card_content.split("\n")
print("\n".join(lines[:30]))
print("...")
print(f"[{len(lines)} lines total]")

## 6. Deliverable D — README.md

The README is the front page of your GitHub repository. 
It is the first thing a hiring manager, colleague, or 
interviewer sees. A great README:
- Explains the problem and why it matters
- Shows key results immediately (with a chart or metric)
- Gives clear setup instructions
- Demonstrates professional communication

In [ ]:
best_f1_str = f"{best_f1:.3f}" if isinstance(best_f1, float) else "N/A"

readme_content = f"""# Social Services Demand Risk Predictor 🇦🇺

> **End-to-end data science project — Australian Public Sector**  
> Predicting welfare demand risk across SA2 regions using open government data.

[![Python](https://img.shields.io/badge/Python-3.11-blue)](https://python.org)
[![License](https://img.shields.io/badge/License-MIT-green)](LICENSE)
[![Data](https://img.shields.io/badge/Data-Open%20Government-orange)](https://data.gov.au)

---

## Project Overview

The Australian Department of Social Services (DSS) supports millions of 
Australians through welfare payments and community services. This project 
asks: **can we proactively identify which communities face the highest 
risk of increased welfare demand?**

Using entirely free, publicly available data from ABS, DSS, and AIHW, 
this project delivers three analytical outputs:

| Task | Technique | Result |
|---|---|---|
| Risk classification | XGBoost + scikit-learn | F1 macro = {best_f1_str} |
| Demand forecasting | Prophet + ARIMA | 4-quarter ahead forecast |
| Policy text analysis | LDA + VADER sentiment | 6 topics tracked 2018–2023 |

---

## Key Finding

High-risk SA2 regions are concentrated in:
- **Northern Territory** — remote communities with high SEIFA disadvantage
- **Remote Western Australia and South Australia** — low IRSD decile regions  
- **Outer suburban corridors** of major cities — population growth + service gaps

SHAP analysis confirmed that **IRSD score** is the single most important 
predictive feature, followed by remoteness classification and population size.

---

## Repository Structure

```
aus-social-services-ds/
│
├── notebooks/
│   ├── 01_problem_framing.ipynb        # Problem statement, data strategy, ethics
│   ├── 02_data_collection.ipynb        # ABS/DSS data ingestion pipelines
│   ├── 03_eda.ipynb                    # Exploratory data analysis (15+ charts)
│   ├── 04_feature_engineering.ipynb    # Feature engineering + preprocessing pipeline
│   ├── 05_modelling.ipynb              # LR → RF → XGBoost + Prophet forecasting
│   ├── 06_nlp_policy_analysis.ipynb    # LDA topic model + VADER sentiment
│   └── 07_executive_report.ipynb       # This notebook — reporting & deployment
│
├── data/
│   ├── raw/          # Source files (not tracked in Git)
│   └── processed/    # Cleaned, feature-engineered outputs
│
├── models/           # Serialised model files (not tracked in Git)
├── app/
│   └── dashboard.py  # Plotly Dash interactive dashboard
├── reports/          # Charts and HTML exports
├── MODEL_CARD.md     # Model documentation and limitations
├── requirements.txt  # Python dependencies
└── README.md
```

---

## Setup & Run

### Prerequisites
- Python 3.11+
- Windows / macOS / Linux
- ~2 GB disk space for datasets

### Installation

```bash
# Clone the repository
git clone https://github.com/YOUR_USERNAME/aus-social-services-ds.git
cd aus-social-services-ds

# Create virtual environment
python -m venv venv
venv\\Scripts\\activate      # Windows
# source venv/bin/activate    # macOS/Linux

# Install dependencies
pip install -r requirements.txt
python -m spacy download en_core_web_sm

# Launch JupyterLab
jupyter lab
```

### Run notebooks in order
Open each notebook in `notebooks/` and run **Run → Run All Cells**:

```
01 → 02 → 03 → 04 → 05 → 06 → 07
```

### Launch the dashboard

```bash
python app/dashboard.py
# Open http://127.0.0.1:8050
```

---

## Data Sources

All data is open, publicly available, and licensed under CC BY 4.0:

| Dataset | Source | URL |
|---|---|---|
| ABS SEIFA 2021 | Australian Bureau of Statistics | [abs.gov.au](https://abs.gov.au) |
| DSS Payment Data | Department of Social Services | [data.gov.au](https://data.gov.au) |
| Regional Population | Australian Bureau of Statistics | [abs.gov.au](https://abs.gov.au) |
| DSS Annual Reports | Department of Social Services | [dss.gov.au](https://dss.gov.au) |

---

## Ethics & Limitations

This project uses only **aggregated, non-personal data** at SA2 region level.
No individual-level data was accessed or used.

See [MODEL_CARD.md](MODEL_CARD.md) for full documentation of model limitations,
ethical considerations, and recommendations for responsible use.

---

## Author

**Firoz Mahmud**  
fmahmud.ruet@gmail.com
Data Analyst
Department of Social Services  

---

## Licence

MIT Licence — see [LICENSE](LICENSE) for details.  
Data sources: CC BY 4.0 (Australian Government Open Data)
"""

readme_path = os.path.join(PROJECT_ROOT, "README.md")
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_content)

print(f"[✓] README.md saved → {readme_path}")
print()
lines = readme_content.split("\n")
print(f"[{len(lines)} lines, {len(readme_content):,} characters]")

## 8. Git Commands to Publish

Run these commands in CMD from your project root to publish to GitHub.

> **Before running:** Create a new repository on github.com called 
> `aus-social-services-ds` (public, no README — we already have one).

In [ ]:
git_commands = f"""
── Step 1: Initialise Git in your project folder ─────────────────────
  cd {PROJECT_ROOT}
  git init
  git branch -M main

── Step 2: Stage all files ───────────────────────────────────────────
  git add .

── Step 3: Check what will be committed ──────────────────────────────
  git status
  (data/raw/ and models/ should NOT appear — they are in .gitignore)

── Step 4: Make your first commit ────────────────────────────────────
  git commit -m "Initial commit: end-to-end DS project — 7 phases complete"

── Step 5: Connect to GitHub ─────────────────────────────────────────
  git remote add origin https://github.com/YOUR_USERNAME/aus-social-services-ds.git

── Step 6: Push to GitHub ────────────────────────────────────────────
  git push -u origin main

── Step 7: Verify on GitHub ─────────────────────────────────────────
  Open https://github.com/YOUR_USERNAME/aus-social-services-ds
  Check: README renders correctly, all 7 notebooks are visible,
         data/raw/ folder does NOT appear

── Step 8: Pin the repo on your profile ─────────────────────────────
  Go to your GitHub profile → Customize profile → Pin repositories
  Pin aus-social-services-ds so it's the first thing visitors see
"""

print(git_commands)

## 9. Final Project Summary

In [ ]:
# Count all deliverables

notebook_count = len([
    f for f in os.listdir(os.path.join(PROJECT_ROOT, "notebooks"))
    if f.endswith(".ipynb")
]) if os.path.exists(os.path.join(PROJECT_ROOT, "notebooks")) else 0

report_count = len([
    f for f in os.listdir(REPORTS_DIR)
]) if os.path.exists(REPORTS_DIR) else 0

model_count = len([
    f for f in os.listdir(MODELS_DIR)
    if f.endswith(".joblib")
]) if os.path.exists(MODELS_DIR) else 0

print("=" * 60)
print("  PROJECT COMPLETE — aus-social-services-ds")
print("=" * 60)
print()
print(f"  Notebooks completed : {notebook_count} / 7")
print(f"  Reports generated   : {report_count} files")
print(f"  Models serialised   : {model_count} .joblib files")
print()

phases = [
    ("Phase 1", "Problem Framing & Data Strategy",
     "Problem statement · data inventory · ethics checklist"),
    ("Phase 2", "Data Collection & Pipelines",
     "ABS/DSS/AIHW ingestion · SQL-style joins · provenance log"),
    ("Phase 3", "Exploratory Data Analysis",
     "15+ charts · target variable created · class balance checked"),
    ("Phase 4", "Feature Engineering",
     "5 new features · StandardScaler pipeline · SMOTE · feature selection"),
    ("Phase 5", "Machine Learning",
     "LR → RF → XGBoost · SHAP explainability · Prophet forecasting"),
    ("Phase 6", "NLP Policy Analysis",
     "LDA topic model · VADER sentiment · TF-IDF distinctive terms"),
    ("Phase 7", "Reporting & Deployment",
     "Executive summary · Dash dashboard · Model card · README · GitHub"),
]

for phase, title, deliverables in phases:
    print(f"  ✓  {phase} — {title}")
    print(f"       {deliverables}")
    print()

print("  DS skills demonstrated:")
skills = [
    "Data engineering (SQL, APIs, pandas pipelines)",
    "Statistical analysis & EDA",
    "Feature engineering & preprocessing",
    "Supervised ML (classification, cross-validation, tuning)",
    "Time-series forecasting (ARIMA, Prophet)",
    "NLP (LDA topic modelling, VADER sentiment, TF-IDF)",
    "Model explainability (SHAP)",
    "Responsible AI (model card, ethics checklist, bias documentation)",
    "Data visualisation (matplotlib, seaborn, plotly)",
    "Dashboard development (Plotly Dash)",
    "Software engineering (virtual environments, Git, .gitignore)",
    "Technical writing (README, executive summary, model card)",
]
for skill in skills:
    print(f"    ✓  {skill}")

print()
print("=" * 60)
print("  This project is ready for your DS job applications.")
print("=" * 60)